## Putting it all together for Training
Pipeline: Preprocessing -> Feature extraction -> spike encoding ->training

In [ ]:
from pipeline_functions import *

import numpy as np

import snntorch as snn
from snntorch import functional as SF

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, TensorDataset, WeightedRandomSampler

#sets seed for random to 42
torch.manual_seed(42) 

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

### Preprocessing

In [ ]:
## preprocessing function to be inputted

preproc_ds = None ##function here

### Feature Extraction

In [ ]:
## feature extraction
fe_X, y = fn_feature_extraction.extractFeatures(preproc_ds)

### Spike Encoding

This is a first place we can alter hyperparameters for results <br>
We can change:
- T_ms
- dt|_ms
- r_min
- r_max

In [ ]:
T_ms = 200
dt_ms = 1
r_min = 50
r_max = 150
encoded_X = spike_encoding.deterministic_rate(fe_X, T_ms=T_ms, dt_ms=dt_ms, r_min=r_min, r_max=r_max)

### Create tensors and dataloaders from the data

Hyperparameter to tune
- batch_size

In [ ]:
batch_size = 128

train_loader, val_loader, test_loader = train.prepare_training_data(encoded_X, y, batch_size=batch_size)

### Module Creation

Hyperparameters:
- hidden_layer
- beta

In [ ]:
# determined from feature extraction
num_features = encoded_X.shape(1)

#tunable
hidden_layer = 316
beta = 0.9

snn = SNNModule.createSNN(num_features, hidden_layer=hidden_layer, beta=beta)

### Training Hyperparameters

Tunable:
- num_epochs (for hyperparameter tuning, keep low to find other parameters optimal, then increase once found for real training)
- lr (learning rate: How fast the model changes)
- criterion: loss function to use, can use either ce_count_loss or ce_rate_loss
- optimizer: method to change weights, can use Adam, SGD w/ momentum, and others. Adam is usually the best but others can be checked
- weight_decay and more: there are other parameters we can change if these dont give good results

In [ ]:
num_epochs = 5
lr = 1e-3

criterion = SF.ce_count_loss()
optimizer = torch.optim.Adam(snn.parameters(), lr=lr)

update_every = 1 # will not change results, just prints updates divisible by this number

history = train.train(model=snn, num_epochs=num_epochs, train_loader=train_loader, val_loader=val_loader,
                      criterion=criterion, optimizer=optimizer, device=device, update_every=update_every, batch_first=True)

### Final Decision

Here we can save our weight parameters to be loaded in

In [ ]:
model_weights = snn.state_dict()

weights_file_path = 'model_weights/snn_weights.pth'

torch.save(model_weights, weights_file_path)